# 1.2 S21 Data Loading

This notebook is split from the original `1_software_feedback_analysis.ipynb`. It loads `s21_data.mat`, separates the |0> and |1> readout traces, and shows the original notebook figures as fixed tutorial references.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_PATHS = [
    Path('./s21_data.mat'),
    Path('../s21_data.mat'),
    Path('../../software/s21_data.mat'),
]

def find_s21_data():
    for p in DATA_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError('Put s21_data.mat in the notebook directory or artery/software/.')

def load_s21():
    import scipy.io as sio
    data_path = find_s21_data()
    read_data = sio.loadmat(data_path)
    read_zero = read_data['data'][0]
    read_one = read_data['data'][1]
    read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
    read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]
    return read_data, read_zero_i, read_zero_q, read_one_i, read_one_q

def demod_part(omega, read_i, read_q, phase=0.0):
    assert read_i.shape == read_q.shape
    ts = np.arange(read_i.shape[1])
    cos_ = np.cos(omega * ts + phase)[None, :]
    sin_ = np.sin(omega * ts + phase)[None, :]
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return np.column_stack([sum_i, sum_q])

OMEGAS = 2 * np.pi * (np.array([6.881, 6.79525, 6.97284]) - 7)

In [ ]:
read_data, read_zero_i, read_zero_q, read_one_i, read_one_q = load_s21()
print('MAT keys:', sorted(read_data.keys()))
print('data shape:', read_data['data'].shape)
print('zero I/Q:', read_zero_i.shape, read_zero_q.shape)
print('one  I/Q:', read_one_i.shape, read_one_q.shape)
print('states:', read_data.get('state'))
print('measurement fidelities:', read_data.get('measure_fids'))

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(np.mean(read_zero_q, axis=0), label='mean Q |0>')
plt.plot(np.mean(read_one_q, axis=0), label='mean Q |1>')
plt.xlabel('sample index')
plt.ylabel('Q amplitude')
plt.legend()
plt.tight_layout()

## Original Notebook Figure: Mean Q Trace

![Original Notebook: Mean Q Trace](../results/1_2_mean_q_trace.png)

## Original Notebook Figure: Mean |0> and |1> Q Traces

![Original Notebook: Mean |0> and |1> Q Traces](../results/1_2_mean_state_q_traces.png)

In [ ]:
energy_zero = np.sum(read_zero_i ** 2, axis=1)
energy_one = np.sum(read_one_i ** 2, axis=1)
print('mean energy |0>:', float(np.mean(energy_zero)))
print('mean energy |1>:', float(np.mean(energy_one)))
print('top energy |0>:', np.sort(energy_zero)[-5:][::-1])
print('top energy |1>:', np.sort(energy_one)[-5:][::-1])

## ARTERY Connection

The raw S21 traces are the measurement record. Hardware receives the same stream sample by sample, so every later module must be written as a streaming datapath rather than as a batch NumPy operation.